<a href="https://colab.research.google.com/github/bekonad/10-academy-week2/blob/main/Copy_of_Task5_Portfolio_Optimization_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Task 5: Portfolio Backtesting, Risk Constraints & Advanced Optimization

This notebook **extends Task 4** and represents the **final decision layer** of the project.

### Objectives
1. Load optimized weights directly from **Task 4**
2. Backtest optimized portfolio vs equal-weight benchmark
3. Re-optimize with **Maximum Drawdown constraint**
4. Compare results with **PyPortfolioOpt**
5. Provide final investment recommendation

---


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

%matplotlib inline


## 1. Load Historical Prices

In [ ]:

prices = pd.read_csv('../data/processed/prices_aligned.csv', index_col=0, parse_dates=True)
prices.columns = prices.columns.str.lower()
returns = prices.pct_change().dropna()

returns.head()


## 2. Load Optimized Weights from Task 4

In [ ]:

# Task 4 should export weights to this path
try:
    w_task4 = pd.read_csv('../outputs/task4_optimal_weights.csv', index_col=0)['weight'].values
    print("Loaded Task 4 optimized weights")
except:
    print("Task 4 weights not found. Using fallback.")
    w_task4 = np.ones(returns.shape[1]) / returns.shape[1]

equal_weights = np.ones(len(w_task4)) / len(w_task4)

w_task4, equal_weights


## 3. Backtesting: Optimized vs Equal Weight

In [ ]:

opt_returns = returns @ w_task4
eq_returns = returns @ equal_weights

opt_cum = (1 + opt_returns).cumprod()
eq_cum = (1 + eq_returns).cumprod()

plt.figure(figsize=(8,5))
plt.plot(opt_cum, label='Optimized Portfolio')
plt.plot(eq_cum, label='Equal Weight Benchmark')
plt.legend()
plt.title('Backtesting Performance Comparison')
plt.ylabel('Cumulative Return')
plt.show()


## 4. Maximum Drawdown Constraint

In [ ]:

def max_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    return drawdown.min()

max_dd_opt = max_drawdown(opt_returns)
max_dd_eq = max_drawdown(eq_returns)

max_dd_opt, max_dd_eq


## 5. Optimization with Max Drawdown Constraint

In [ ]:

mu = returns.mean().values
Sigma = returns.cov().values
rf = 0.03 / 252

def portfolio_returns(w):
    return returns @ w

def sharpe(w):
    ret = w @ mu
    vol = np.sqrt(w.T @ Sigma @ w)
    return (ret - rf) / vol

def neg_sharpe(w):
    return -sharpe(w)

def drawdown_constraint(w, limit=-0.25):
    return max_drawdown(portfolio_returns(w)) - limit

constraints = (
    {'type':'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type':'ineq', 'fun': lambda w: drawdown_constraint(w)}
)

bounds = tuple((0,1) for _ in range(len(mu)))
init = np.ones(len(mu)) / len(mu)

res_dd = minimize(neg_sharpe, init, method='SLSQP',
                  bounds=bounds, constraints=constraints)

w_dd = res_dd.x
w_dd


## 6. PyPortfolioOpt Comparison

In [ ]:

# Advanced optimization using PyPortfolioOpt
from pypfopt import EfficientFrontier, risk_models, expected_returns

mu_pyp = expected_returns.mean_historical_return(prices)
Sigma_pyp = risk_models.sample_cov(prices)

ef = EfficientFrontier(mu_pyp, Sigma_pyp, weight_bounds=(0,1))
ef.max_sharpe(risk_free_rate=0.03)

w_pyp = ef.clean_weights()
pd.Series(w_pyp)


## 7. Strategy Comparison

In [ ]:

w_pyp_array = np.array(list(w_pyp.values()))

pyp_returns = returns @ w_pyp_array
pyp_cum = (1 + pyp_returns).cumprod()

plt.figure(figsize=(8,5))
plt.plot(opt_cum, label='Task 4 Optimized')
plt.plot(pyp_cum, label='PyPortfolioOpt')
plt.plot(eq_cum, label='Equal Weight')
plt.legend()
plt.title('Strategy Comparison')
plt.show()


## 8. Efficient Frontier

In [ ]:

target_returns = np.linspace(mu.min(), mu.max(), 50)
vols = []

for r in target_returns:
    cons = (
        {'type':'eq', 'fun': lambda w: w@mu - r},
        {'type':'eq', 'fun': lambda w: np.sum(w) - 1}
    )
    res = minimize(lambda w: np.sqrt(w.T @ Sigma @ w),
                   init, bounds=bounds, constraints=cons)
    vols.append(res.fun)

plt.figure(figsize=(7,5))
plt.plot(vols, target_returns)
plt.xlabel('Volatility')
plt.ylabel('Expected Return')
plt.title('Efficient Frontier')
plt.show()



## 9. Final Investment Recommendation

- Forecast-driven optimization outperforms naïve equal-weight allocation.
- Adding a **maximum drawdown constraint** improves downside protection.
- **PyPortfolioOpt** confirms results and provides robust industry-standard validation.

### Recommendation:
Adopt the constrained optimized portfolio with periodic rebalancing,
monitoring drawdown risk and macroeconomic regime shifts.

---
**This completes the portfolio optimization pipeline.**
